Make the voxelised electron density

In [5]:
!python 1_voxelize_pdb.py --topology "data/1crn.psf" --trajectory "data/1crn_eq_aligned.dcd" --bbmax 41 --save_path data --system "1crn"

Using MD trajectory mode
Using data/1crn.psf as reference PDB
Using data/1crn_eq_aligned.dcd as trajectory
bbmax: 41.0
n_voxels: 52
cutoff: 5.0
pdb_path: None
trajectory: data/1crn_eq_aligned.dcd
reference_pdb: None
topology: data/1crn.psf
save_path: data
system: 1crn
selection: protein and not element H
creating universe...
/nobackup/proj/disk/naiss2025-23-641/shared/cardoch/xssdense/.venv/lib/python3.12/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"
done.
bounding box: 41.0 Å, num. of voxels: 52, voxel size: 1.61 Å
Computing density for each frame...
done./10000
Saving density to data/1crn_41bb_1d61dv_

Tranform densities to tfrecord for fast training

In [6]:
!python 2_generate_tfrecord.py  --h5 "data/1crn_41bb_1d61dv_5co.h5" --output_dir "data/1crn_41bb_1d61dv_5co.tfrecord"

2026-11-10 11:06:10.678487: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-10 11:06:10.687427: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-10 11:06:10.691587: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
h5: data/1crn_41bb_1d61dv_5co.h5
output_dir: data/1crn_41bb_1d61dv_5co.tfrecord
block: 100
test_every: 10
num_proc: 8
Loaded 10000 frames of (52, 52, 52), voxel size 1.6100 Å, bounding box 41.0 Å
Electrons per frame: mean 2205, rel. std 0.00%
Split: 9000 train, 1000 test
Normalization: min 0, max 1.57592
Writing test.tfrecords: 100%|████████████| 1000/1000 [00:00

Do the training

In [7]:
!python 3_train_VAE.py  --datapath "data/1crn_41bb_1d61dv_5co.tfrecord" --savepath "data" --epochs=30 --mode constant --beta_input 1e-1 --batchsize 64

GPU DETECTED: TensorFlow found 1 GPU(s):
  GPU 0: /physical_device:GPU:0
datapath: data/1crn_41bb_1d61dv_5co.tfrecord
train_datapath: data/1crn_41bb_1d61dv_5co.tfrecord/tfrecords/train.tfrecords
test_datapath: data/1crn_41bb_1d61dv_5co.tfrecord/tfrecords/test.tfrecords
meta_datapath: data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json
savepath: data
mode: constant
beta_input: 0.1
z_dim: 8
learning_rate: 0.0003
epochs: 30
batchsize: 64
patience: 5
min_improvement: 0.005
save_every_epoch: False
split: 8100 train, 900 validation

Epoch 1/30
Step 0, Loss: 657.63251, MSE: 117.14986, KL: 0.08842
Step 10, Loss: 655.92542, MSE: 116.85270, KL: 0.13993
Step 20, Loss: 653.15308, MSE: 116.35565, KL: 0.94103
Step 30, Loss: 644.76416, MSE: 115.25242, KL: 5.64951
Step 40, Loss: 649.85016, MSE: 125.86294, KL: 24.47078
Step 50, Loss: 453.84265, MSE: 102.70715, KL: 143.64841
Step 60, Loss: 353.08752, MSE: 82.68428, KL: 74.56691
Step 70, Loss: 311.51050, MSE: 72.55341, KL: 39.02422
Step 80, Loss: 307.53162

Compute scattering from voxel density, split into two classes, and save as the reference we want to reconstruct

In [11]:
!python helper/crambin_saxs_class.py --h5 data/1crn_41bb_1d61dv_5co.h5 --out_dir data --rho_bulk 0 --padding 3 --meta data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json

h5: data/1crn_41bb_1d61dv_5co.h5
out_dir: data
padding: 3
n_pca: 2
n_clusters: 2
dark_index: 0
rho_bulk: 0.0
drho: 0.0
shell: 0.0
pa_yield: 1.0
fig_dir: data/figures
meta: data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json
volume_per_electron: 2.6
1000 of 10000 frames are test frames
frame 10000/10000
saved data/1crn_41bb_1d61dv_5co_xss.h5
class #1: 4479 frames, distance to dark curve 0.4137
class #2: 5521 frames, distance to dark curve 0.6623
saved data/1crn_41bb_1d61dv_5co_xss_pca.h5
saved data/figures/curves_norm.png
saved data/figures/pca_clusters.png
saved data/figures/class_intensity.png
saved data/synthetic_class1_vacuum
saved data/synthetic_class2_vacuum
saved data/figures/synthetic_deltaI.png


Run the reconstruction for class 1 and 2

In [12]:
!python 4_genetic_algorithm.py \
    --model_path data/1crn_41bb_1d61dv_5co.vae \
    --meta data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json \
    --h5 data/1crn_41bb_1d61dv_5co.h5 \
    --dI_curve data/synthetic_class1_vacuum/deltaI.csv \
    --dark_curve data/synthetic_class1_vacuum/ground_abs.csv \
    --rho_bulk 0 \
    --target_yield 1 \
    --output_folder results/synthetic_class1_vacuum \
    --padding 3 \
    --qmax 0.6

model_path: data/1crn_41bb_1d61dv_5co.vae
meta: data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json
h5: data/1crn_41bb_1d61dv_5co.h5
dark_index: 0
dark_curve: data/synthetic_class1_vacuum/ground_abs.csv
dI_curve: data/synthetic_class1_vacuum/deltaI.csv
output_folder: results/synthetic_class1_vacuum
rho_bulk: 0.0
target_yield: 1.0
yield_weight: 2.0
qmax: 0.6
padding: 3
pop_size: 64
generations: 80
n_elite: 5
n_random: 20
mutation_rate: 0.3
top_k: 5
volume_per_electron: 2.6
seed: 0
Envelope: 1374 voxels (5734 Å³) for 2205 electrons
Dark fit: scale 1.0000, drho 0.0000, shell 0.0 Å, r2 1.00000
Gen   0  best 0.0511  r2 0.0511  yield 1.000
Gen   1  best 0.6332  r2 0.8532  yield 0.668
Gen   2  best 0.7016  r2 0.8752  yield 0.705
Gen   3  best 0.9291  r2 0.9366  yield 0.939
Gen   4  best 0.9291  r2 0.9366  yield 0.939
Gen   5  best 0.9314  r2 0.9466  yield 0.913
Gen   6  best 0.9314  r2 0.9466  yield 0.913
Gen   7  best 0.9431  r2 0.9702  yield 0.884
Gen   8  best 0.9503  r2 0.9503  yield 1.000


In [13]:
!python 4_genetic_algorithm.py \
    --model_path data/1crn_41bb_1d61dv_5co.vae \
    --meta data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json \
    --h5 data/1crn_41bb_1d61dv_5co.h5 \
    --dI_curve data/synthetic_class2_vacuum/deltaI.csv \
    --dark_curve data/synthetic_class2_vacuum/ground_abs.csv \
    --rho_bulk 0 \
    --target_yield 1 \
    --output_folder results/synthetic_class2_vacuum \
    --padding 3 \
    --qmax 0.6

model_path: data/1crn_41bb_1d61dv_5co.vae
meta: data/1crn_41bb_1d61dv_5co.tfrecord/meta/meta.json
h5: data/1crn_41bb_1d61dv_5co.h5
dark_index: 0
dark_curve: data/synthetic_class2_vacuum/ground_abs.csv
dI_curve: data/synthetic_class2_vacuum/deltaI.csv
output_folder: results/synthetic_class2_vacuum
rho_bulk: 0.0
target_yield: 1.0
yield_weight: 2.0
qmax: 0.6
padding: 3
pop_size: 64
generations: 80
n_elite: 5
n_random: 20
mutation_rate: 0.3
top_k: 5
volume_per_electron: 2.6
seed: 0
Envelope: 1374 voxels (5734 Å³) for 2205 electrons
Dark fit: scale 1.0000, drho 0.0000, shell 0.0 Å, r2 1.00000
Gen   0  best -0.4108  r2 -0.4108  yield 1.000
Gen   1  best 0.3729  r2 0.3729  yield 1.000
Gen   2  best 0.6229  r2 0.6229  yield 1.000
Gen   3  best 0.8110  r2 0.8110  yield 1.000
Gen   4  best 0.8110  r2 0.8110  yield 1.000
Gen   5  best 0.8115  r2 0.8115  yield 1.000
Gen   6  best 0.8110  r2 0.8110  yield 1.000
Gen   7  best 0.9257  r2 0.9257  yield 1.000
Gen   8  best 0.9257  r2 0.9257  yield 1.00